# Task 06 | D2 × E3 

**Randomized-network POINT Identification — 独立学生任务 Demo**

> 只进行总体因果识别，不训练模型、不估计效果数值，也不读取 outcome ground truth。遵循顺序：**业务问题 → setting → support → POINT 推导 → IdentificationResult**。最后增加穷举对拍与失败场景。

## 1. 业务问题与目标

公司有 $N$ 位销售员，合作网络在实验前固定。公司从所有销售员中**恰好选出 $m$ 人**开放 AI 助手（D2）。每个销售员 $i$ 有处理前确定的核心合作伙伴 $C_i\subseteq\mathcal N(i)$；只有**核心伙伴严格过半**有 AI，才令 $S_i=1$（E3）。

我们不把 $S_i$ 假设为决定结果的充分状态，而是对满足相同查询 $(T_i,S_i)=(t,s)$ 的完整处理分配依设计权重取平均，得到 $\mu_i(t,s)$。目标是确定这些均值是否具有由总体观测分布唯一确定的 **POINT functional**。

**与 shared baseline 不同：**（1）固定处理人数引入处理之间的依赖；（2）暴露由预先选择的核心伙伴而非全部邻居决定。

## 2. Setting：处理前图、核心伙伴和完全随机化

构造可复核的 $N=8,m=3$ 玩具网络：每个节点与距离为 1 或 2 的圆环节点相连（4 个合作邻居）；核心伙伴只选择 3 人。这里网络和核心集合都是预先确定的，不依赖处理结果。

完整合法分配空间：

$$\Omega_m=\{\mathbf u\in\{0,1\}^N:\textstyle\sum_j u_j=m\},\qquad q(\mathbf u)=\frac{\mathbf1\{\mathbf u\in\Omega_m\}}{\binom Nm}.$$

暴露规则：$K_i=\sum_{j\in C_i}T_j$，$S_i=\mathbf1\{K_i>\lfloor h_i/2\rfloor\}$。

In [1]:
from __future__ import annotations
from fractions import Fraction
from itertools import combinations
from math import comb
from pathlib import Path
import sys
from IPython.display import display, Markdown

# Run All from the task folder OR from the upstream repository root.
if not Path("support.py").is_file():
    folder = Path.cwd() / "student_work" / "task06"
    if not folder.joinpath("support.py").is_file():
        raise FileNotFoundError("Please run from task06/ or the repository root")
    sys.path.insert(0, str(folder))

from support import Task06Setting, support_for_node, support_probabilities
from rule import REQUIRED_ASSUMPTIONS, identify_arm, identify_all

N, m = 8, 3
A = [[0 for j in range(N)] for i in range(N)]
for i in range(N):
    for step in (1, 2):
        j = (i + step) % N
        A[i][j] = A[j][i] = 1

C = {i: [(i - 1) % N, (i + 1) % N, (i + 2) % N] for i in range(N)}
setting = Task06Setting.create(A, C, m)
print(f"N={setting.n}, m={setting.m}, total legal assignments = C(N,m)={comb(N,m)}")
print(f"Node 0 full neighbors = {[j for j in range(N) if A[0][j]]}")
print(f"Node 0 core partners  = {setting.core_partners[0]}")
print(f"Node 0 degree = {sum(A[0])}, core size h_0 = {len(setting.core_partners[0])}")
assert len(setting.core_partners[0]) < sum(A[0])


N=8, m=3, total legal assignments = C(N,m)=56
Node 0 full neighbors = [1, 2, 6, 7]
Node 0 core partners  = (1, 2, 7)
Node 0 degree = 4, core size h_0 = 3


### 假设的**显式**确认

要把目标识别为 POINT，研究者必须基于实验协议明确接受下列假设；对真实数据，不能因某个函数返回 `True` 就认为假设得到了统计证明。

1. `fixed_pretreatment_network`：合作网络在处理前固定；
2. `fixed_pretreatment_core_partners`：每个核心集合 $C_i$ 在处理前固定且合法；
3. `known_complete_randomization`：确知所有恰好 $m$ 人接受处理的向量等概率；
4. `full_vector_randomization`：给定处理前信息，完整处理向量与潜在结果族独立；
5. `consistency`：$Y_i^{obs}=Y_i(\mathbf T)$；
6. `finite_conditional_first_moments`：目标涉及的潜在结果条件一阶绝对矩有限。


In [2]:
assumptions = {name: True for name in REQUIRED_ASSUMPTIONS}
print("Explicitly admitted assumptions for this illustrative design:")
for name in REQUIRED_ASSUMPTIONS:
    print("  [admitted]", name)


Explicitly admitted assumptions for this illustrative design:
  [admitted] fixed_pretreatment_network
  [admitted] fixed_pretreatment_core_partners
  [admitted] known_complete_randomization
  [admitted] full_vector_randomization
  [admitted] consistency
  [admitted] finite_conditional_first_moments


## 3. Support：计算 $\pi_i(t,s)$，不能只声称 positivity

固定本人处理 $T_i=t$ 后，剩余 $N-1$ 人需要选出 $R=m-t$ 名接受处理；其中 $h_i$ 人是核心伙伴，$L=N-1-h_i$ 人不是。因此

$$K_i\mid T_i=t\sim\operatorname{Hypergeom}(N-1,h_i,m-t).$$

联合计数为

$$P(T_i=t,K_i=k\mid A,X,C)=\frac{\binom{h_i}{k}\binom{N-1-h_i}{m-t-k}}{\binom Nm}.$$

令 $I_0=\{0,\ldots,\lfloor h_i/2\rfloor\}$，$I_1=\{\lfloor h_i/2\rfloor+1,\ldots,h_i\}$，则

$$\pi_i(t,s)=\frac{\sum_{k\in I_s}\binom{h_i}{k}\binom{N-1-h_i}{m-t-k}}{\binom Nm}.$$

公式实现放在 `support.py`。精确分数由 `Fraction` 表示，没有浮点舍入误差。

In [3]:
pi = support_for_node(setting, 0)
rows = [
    f"| ({t},{s}) | {pi[t,s]} | {float(pi[t,s]):.6f} | {pi[t,s] > 0} |"
    for t in (0, 1) for s in (0, 1)
]
display(Markdown("| Arm (t,s) | 精确支持概率 | 十进制 | 可达 |\n"
                 "|:---:|---:|---:|:---:|\n" + "\n".join(rows)))
expected = {(0,0):Fraction(22,56), (0,1):Fraction(13,56),
            (1,0):Fraction(18,56), (1,1):Fraction(3,56)}
assert pi == expected
assert sum(pi.values(), Fraction(0)) == 1
assert pi[1,0]+pi[1,1] == Fraction(m,N)
assert pi[0,0]+pi[0,1] == Fraction(N-m,N)
print("Analytical probabilities match (22, 13, 18, 3)/56; all marginals checked.")


| Arm (t,s) | 精确支持概率 | 十进制 | 可达 |
|:---:|---:|---:|:---:|
| (0,0) | 11/28 | 0.392857 | True |
| (0,1) | 13/56 | 0.232143 | True |
| (1,0) | 9/28 | 0.321429 | True |
| (1,1) | 3/56 | 0.053571 | True |

Analytical probabilities match (22, 13, 18, 3)/56; all marginals checked.


### Population zero-support 的充要条件

对一个节点，$h_i=|C_i|$，令 $b=\lfloor h_i/2\rfloor$，$R=m-t$。合法的核心伙伴处理人数满足

$$k_{min}=\max(0,R-(N-1-h_i)),\quad k_{max}=\min(h_i,R).$$

由整数区间交集可得（已包含本人状态可行性）：

$$\begin{aligned}
\pi_i(t,0)>0&\iff 0\le m-t\le N-1-\lceil h_i/2\rceil,\\
\pi_i(t,1)>0&\iff \lfloor h_i/2\rfloor+1\le m-t\le N-1.
\end{aligned}$$

例如 $N=6,m=2,h_i=3$：当 $T_i=1$，其他人最多一人获得 AI，核心伙伴不可能严格过半，所以 $\pi_i(1,1)=0$。

## 4. POINT：从潜在结果到可观测 functional

对所有具有正支持的 $(i,t,s)$，定义

$$\mathcal U_{i,t,s}=\{\mathbf u\in\Omega_m:u_i=t,S_i(\mathbf u)=s\},$$

$$\mu_i(t,s)=\sum_{\mathbf u\in\mathcal U_{i,t,s}}\frac{q(\mathbf u)}{\pi_i(t,s)}\,\mathbb E[Y_i(\mathbf u)\mid A,X,C].$$

**证明**（每一步对应明确假设）：

$$\begin{aligned}
&\mathbb E[Y_i^{obs}\mid A,X,C,T_i=t,S_i=s]\\
&\overset{\mathrm{iterated\ expectation}}{=}
\sum_{\mathbf u\in\mathcal U_{i,t,s}}
\frac{q(\mathbf u)}{\pi_i(t,s)}
\mathbb E[Y_i^{obs}\mid A,X,C,\mathbf T=\mathbf u]\\
&\overset{\mathrm{consistency}}{=}
\sum_{\mathbf u\in\mathcal U_{i,t,s}}
\frac{q(\mathbf u)}{\pi_i(t,s)}
\mathbb E[Y_i(\mathbf u)\mid A,X,C,\mathbf T=\mathbf u]\\
&\overset{\mathrm{randomization}}{=}
\sum_{\mathbf u\in\mathcal U_{i,t,s}}
\frac{q(\mathbf u)}{\pi_i(t,s)}
\mathbb E[Y_i(\mathbf u)\mid A,X,C]
=\mu_i(t,s).
\end{aligned}$$

右侧为**总体观测分布的条件均值**，故任何两个满足当前假设、具有相同总体观测分布的因果世界都给出相同的 $\mu_i(t,s)$，即 POINT。

**边界：** 这不是一组数值估计，也**没有**假设 $S_i$ 充分代表所有干扰。D2 下不同本人 treatment 会改变其他人可分配名额，因此跨 arm 对比是设计特定的因果对比，并非一般意义上保持所有他人处理不变的单人 intervention。

## 5. IdentificationResult：按 arm 返回 POINT 或拒绝

`rule.py` 仅输出本任务的本地轻量 `IdentificationResult`，不伪装成主仓库共用 `pfn_pipeline` 的识别证书。正支持且假设显式接受时，返回可观测泛函；否则给出 `UNSUPPORTED` 与明确的原因码。

In [4]:
results = identify_all(setting, assumptions=assumptions)
for (t, s), result in results[0].items():
    print(f"node 0, arm=({t},{s}), pi={result.support}, status={result.status}")
    print("  functional:", result.identified_functional)
    assert result.status == "POINT"
print(f"All {4*N} positive-support arm queries are POINT in the illustrative N=8 design.")


node 0, arm=(0,0), pi=11/28, status=POINT
  functional: mu_0(0,0) = E[Y_0^obs | A, X, C, T_0=0, S_0=0]
node 0, arm=(0,1), pi=13/56, status=POINT
  functional: mu_0(0,1) = E[Y_0^obs | A, X, C, T_0=0, S_0=1]
node 0, arm=(1,0), pi=9/28, status=POINT
  functional: mu_0(1,0) = E[Y_0^obs | A, X, C, T_0=1, S_0=0]
node 0, arm=(1,1), pi=3/56, status=POINT
  functional: mu_0(1,1) = E[Y_0^obs | A, X, C, T_0=1, S_0=1]
All 32 positive-support arm queries are POINT in the illustrative N=8 design.


## 6. 独立 brute-force：枚举所有合法完整分配（仅 N≤8）

独立写一个**不调用支持概率公式**的枚举 oracle：直接遍历所有 $\binom Nm$ 个 treated 集合，按 $q(\mathbf u)=1/\binom Nm$ 给每个节点的 $(T_i,S_i)$ 计数，再与解析组合公式对拍。

注意这里只计算随机化设计概率，不读取或构造任何 outcome truth。

In [5]:
def brute_force_support(setting):
    counts = {i: {(t,s):0 for t in (0,1) for s in (0,1)}
              for i in range(setting.n)}
    for treated_tuple in combinations(range(setting.n), setting.m):
        treated = set(treated_tuple)
        for i in range(setting.n):
            t = int(i in treated)
            k = sum(int(j in treated) for j in setting.core_partners[i])
            s = int(2*k > len(setting.core_partners[i]))
            counts[i][(t,s)] += 1
    denominator = comb(setting.n, setting.m)
    return {i: {arm:Fraction(count,denominator)
                for arm,count in counts_i.items()}
            for i,counts_i in counts.items()}

oracle = brute_force_support(setting)
formula = support_probabilities(A, C, m)
assert oracle == formula
print(f"Brute-force vs analytic: EXACT MATCH for {N} nodes x 4 arms, "
      f"across {comb(N,m)} legal full assignments.")
print("Node 0 arm counts:", {arm: int(p*comb(N,m)) for arm,p in oracle[0].items()})


Brute-force vs analytic: EXACT MATCH for 8 nodes x 4 arms, across 56 legal full assignments.
Node 0 arm counts: {(0, 0): 22, (0, 1): 13, (1, 0): 18, (1, 1): 3}


## 7. 失败场景：零支持 / 未确认假设 / 无效输入

- **零支持**：因为固定总名额，某些 arm 在总体上永不可能发生，应拒绝（不是继续应用 POINT）。
- **未确认假设**：即使有正支持，也不自动当作 POINT，更不能自动视为 PARTIAL。
- **非法输入**：例如 E3 核心伙伴集合为空或包含非邻居，应在构造 setting 时直接报错。

In [6]:
N2, m2 = 6, 2
A2 = [[int(i!=j) for j in range(N2)] for i in range(N2)]
C2 = {i: [(i+j)%N2 for j in (1,2,3)] for i in range(N2)}
zero_setting = Task06Setting.create(A2, C2, m2)
zero = identify_arm(zero_setting, 0, 1, 1, assumptions=assumptions)
assert zero.support == 0
assert zero.status == "UNSUPPORTED" and zero.identified_functional is None
assert zero.reason_code == "ZERO_POPULATION_SUPPORT"
print("Zero-support:", zero.status, zero.reason_code)

unconfirmed = identify_arm(setting, 0, 1, 1)
assert unconfirmed.support > 0
assert unconfirmed.status == "UNSUPPORTED"
assert unconfirmed.reason_code == "REQUIRED_ASSUMPTIONS_NOT_ADMITTED"
print("Unconfirmed assumptions:", unconfirmed.status, unconfirmed.reason_code)

try:
    bad_C = dict(C)
    bad_C[0] = []
    Task06Setting.create(A, bad_C, m)
except ValueError as err:
    print("Invalid E3 core set rejected:", str(err))
else:
    raise AssertionError("Invalid E3 core set was not rejected")


Zero-support: UNSUPPORTED ZERO_POPULATION_SUPPORT
Unconfirmed assumptions: UNSUPPORTED REQUIRED_ASSUMPTIONS_NOT_ADMITTED
Invalid E3 core set rejected: C_0 must be nonempty (E3 requires h_i>0)


## 8. 结论与运行说明

- **数学对象**：设计标准化 $\mu_i(t,s)$，不是模拟器真实结果，也不要求 exposure sufficiency。
- **Support**：固定名额的组合数计数 / 条件超几何分布；不是独立 Bernoulli 的乘积。
- **识别**：正支持、已知设计、完整向量随机化与一致性给出 $\mu_i(t,s)=E[Y_i^{obs}\mid A,X,C,T_i=t,S_i=s]$。
- **异常**：零支持拒绝，未确认假设拒绝，不直接宣称 PARTIAL。
- **验证**：已在本 Notebook 中按合法完整分配穷举对拍；更多 N≤8 系统测试参见 `test_rule.py`。
